In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/india_agriculture_seed_sales_data.csv')


Annual Profit Trend

In [9]:


annual = df.groupby('Year').agg(
    Avg_Margin=('Profit_Margin_%','mean'),
    Total_Revenue=('Total_Revenue','sum'),
    Total_Cost=('Total_Cost','sum')
).round(2)
annual['Profit_Percentage(%)'] = ((annual['Total_Revenue'] - annual['Total_Cost']) /annual['Total_Revenue'] * 100).round(2)
print(annual)

      Avg_Margin  Total_Revenue    Total_Cost  Profit_Percentage(%)
Year                                                               
2021        6.72   1.030771e+09  8.690641e+08                 15.69
2022        4.09   9.459111e+08  8.165334e+08                 13.68
2023        1.56   8.550009e+08  7.575114e+08                 11.40
2024       -2.45   7.616881e+08  6.965872e+08                  8.55
2025       -6.45   6.787777e+08  6.426806e+08                  5.32


2021 profit percentage ~16% → 2025 profit percentage ~5% — confirms business problem

Region × Crop Profitability Matrix

In [10]:
pivot = df.pivot_table(
    values='Profit_Margin_%',
    index='Region',
    columns='Vegetable_Type',
    aggfunc='mean'
).round(1)
print(pivot)

Vegetable_Type  Bitter Gourd  Bottle Gourd  Brinjal  Cabbage  Carrot  \
Region                                                                 
Central                 -5.1         -12.3     -0.5     -8.8    -1.1   
East                    -4.6          -7.1      0.3     -3.0     1.0   
North                   -2.8          -9.1     -2.4     -8.9    -2.4   
South                    0.4          -5.9     10.4     -0.3     2.9   
West                    -4.3         -12.0     -2.0     -9.6    -5.5   

Vegetable_Type  Cauliflower  Chili  Coriander  Cucumber  Fenugreek  Garlic  \
Region                                                                       
Central                -8.5    6.5        9.2      -8.4        6.2    12.6   
East                   -2.9    7.7        8.1      -3.8        5.1    14.4   
North                  -6.6    8.0        9.5      -6.9        5.9    11.7   
South                  -1.3   10.7       11.5      -2.5        8.9    15.5   
West                   -8.0

Loss-Making Segments

In [13]:
loss = df[df['Profit_Margin_%'] < 0].groupby(
    ['Region','Vegetable_Type']
)['Profit_Margin_%'].mean().round(2).sort_values().head(10)
print("Top 10 Loss-Making Segments(region + Vagetable):")
print(loss)

Top 10 Loss-Making Segments(region + Vagetable):
Region   Vegetable_Type
West     Radish           -31.15
Central  Radish           -31.12
East     Radish           -28.62
North    Radish           -28.56
         Pumpkin          -28.28
         Spinach          -27.48
West     Pumpkin          -26.71
Central  Bottle Gourd     -26.66
         Pumpkin          -26.61
         Cucumber         -26.37
Name: Profit_Margin_%, dtype: float64


Save for Power BI

In [14]:
df['Is_Loss_Making'] = df['Profit_Margin_%'] < 0
df['Revenue_Category'] = pd.cut(df['Total_Revenue'],
    bins=[0, 50000, 150000, 500000, float('inf')],
    labels=['Low','Medium','High','Very High'])